# Grupo de lote de cada linha — Estrutural, Articulação Regional, Distribuição Local

Classifica cada linha de ônibus (`linha_blt`) no **grupo de lote** do edital de concessão:
**Estrutural** (lotes `E1`–`E9`), **Articulação Regional** (`AR0`–`AR9`) e **Distribuição Local**
(`D1`–`D13`). É a mesma tipologia usada nas análises do IQT (`Dados/Análise IQT/`).

A fonte é a própria série oficial de passageiros transportados
(`../outputs/01/Dados_Diario_Oficial_2023_2026.parquet`), que traz `Grupo`, `Lote` e `Empresa` em
cada linha × dia — as mesmas colunas da planilha diária de 02/09/2026 que veio junto da análise do
IQT. Não há uma tabela linha → lote separada.

Como a classificação é por **linha**, ela alimenta os recortes que dependem de linha (a oferta existe
só por linha) e permite ler o sinal Tarifa Zero por tipo de serviço. Cada linha recebe o grupo
**dominante em volume** em 2023–2024, a janela da bilhetagem. Linhas que trocaram de grupo no período
(reestruturação de lotes em 2024) ficam marcadas em `mudou_grupo`.

Saída: `../outputs/01/grupo_linha.parquet`.

## Seção 0 — imports e configuração

In [1]:
import re
from pathlib import Path

import pandas as pd
import polars as pl

ANOS = ["2023", "2024"]       # janela da bilhetagem: a classificacao vale para esses anos
LIMIAR_MUDOU = 0.90           # linha com menos que isto do volume no grupo dominante = "mudou de grupo"

DIR_SAIDA = Path("../outputs/01")
OFICIAL = DIR_SAIDA / "Dados_Diario_Oficial_2023_2026.parquet"
DIR_IQT = Path("../Dados/Análise IQT")
SNAPSHOT_IQT = DIR_IQT / "02SETEMBRO2026.xlsx"
PLANILHA_IQT = DIR_IQT / "Representação TCM - Planilha Histórico Mensal do IQT.xlsx"

# A grafia do Grupo varia entre anos (ARTICULAÇÃO / ARTICULACAO, DISTRIBUIÇÃO / DISTRIBUICAO):
# normaliza para caixa alta sem acento antes de mapear. "OUTRO PC" (servicos fora da concessao
# regular, ~0,1% do volume) nao tem grupo e fica fora.
GRUPOS = {
    "GRUPO ESTRUTURAL": "Estrutural",
    "GRUPO ARTICULACAO": "Articulação Regional",
    "GRUPO DE DISTRIBUICAO": "Distribuição Local",
}
ROTULOS_GRUPO = list(GRUPOS.values())
# Conferencia independente do Grupo: o prefixo do codigo do lote.
PREFIXO_LOTE = {"E": "Estrutural", "AR": "Articulação Regional", "D": "Distribuição Local"}

## Seção 1 — grupo, lote e empresa dominantes por linha

Para cada `linha_blt` soma-se o volume (`of_total`) de 2023–2024 por grupo e fica-se com o maior. O
lote e a empresa são os dominantes **dentro desse grupo**. `share_grupo_dominante` é a parcela do
volume da linha que está no grupo escolhido — perto de 1 para quase todas.

Linhas sem `linha_blt` (o serviço `PAESE` e afins, que não casam com a bilhetagem) ficam fora.

In [2]:
of = (
    pl.read_parquet(OFICIAL, columns=["Grupo", "Lote", "Empresa", "linha_blt", "Ano", "of_total"])
    .filter(pl.col("linha_blt").is_not_null() & pl.col("Ano").is_in(ANOS))
    .with_columns(
        pl.col("Grupo").str.to_uppercase().str.replace_all("Ç", "C").str.replace_all("Ã", "A")
          .replace_strict(GRUPOS, default=None).alias("grupo"),
        pl.col("Lote").str.strip_chars().alias("lote"),
        pl.col("Empresa").str.replace_all(r"\s+", " ").str.strip_chars().alias("empresa"),
    )
)
sem_grupo = of.filter(pl.col("grupo").is_null())
print(f"registros linha x dia sem grupo (OUTRO PC etc.): {sem_grupo.height:,} "
      f"({100 * sem_grupo['of_total'].sum() / of['of_total'].sum():.2f}% do volume) -- ficam fora")
x = of.filter(pl.col("grupo").is_not_null()).to_pandas()

# volume por linha x grupo; grupo dominante = o de maior volume
vol_grupo = x.groupby(["linha_blt", "grupo"], as_index=False)["of_total"].sum()
vol_linha = vol_grupo.groupby("linha_blt")["of_total"].sum().rename("vol_linha")
dom = (
    vol_grupo.sort_values("of_total", ascending=False).drop_duplicates("linha_blt")
    .rename(columns={"of_total": "vol_dominante"})
    .merge(vol_linha, on="linha_blt")
)
dom["share_grupo_dominante"] = dom["vol_dominante"] / dom["vol_linha"]
dom["mudou_grupo"] = dom["share_grupo_dominante"] < LIMIAR_MUDOU

# lote e empresa dominantes DENTRO do grupo dominante
def dominante_dentro(coluna):
    v = (x.merge(dom[["linha_blt", "grupo"]], on=["linha_blt", "grupo"])
           .groupby(["linha_blt", coluna], as_index=False)["of_total"].sum())
    return v.sort_values("of_total", ascending=False).drop_duplicates("linha_blt")[["linha_blt", coluna]]

grupo_linha = (
    dom.merge(dominante_dentro("lote"), on="linha_blt")
       .merge(dominante_dentro("empresa"), on="linha_blt")
       .sort_values("linha_blt").reset_index(drop=True)
)
grupo_linha["grupo"] = pd.Categorical(grupo_linha["grupo"], categories=ROTULOS_GRUPO, ordered=True)

resumo = grupo_linha.groupby("grupo", observed=True).agg(
    linhas=("linha_blt", "size"), vol=("vol_linha", "sum"), mudaram=("mudou_grupo", "sum"))
resumo["pct_volume"] = 100 * resumo["vol"] / resumo["vol"].sum()
print(f"\nlinhas classificadas: {len(grupo_linha)}")
print(resumo.drop(columns="vol").round(1).to_string())

registros linha x dia sem grupo (OUTRO PC etc.): 730 (0.08% do volume) -- ficam fora



linhas classificadas: 1469
                      linhas  mudaram  pct_volume
grupo                                            
Estrutural               440       11        32.8
Articulação Regional     470        4        24.3
Distribuição Local       559       11        42.9


### Diagnóstico — a classificação é estável?

Três checagens: quanto do volume está fora do grupo dominante da própria linha, quais linhas
trocaram de grupo no período, e se o prefixo do código do lote (`E`/`AR`/`D`) concorda com o grupo.

In [3]:
fora = 1 - grupo_linha["vol_dominante"].sum() / grupo_linha["vol_linha"].sum()
print(f"volume fora do grupo dominante da propria linha: {100 * fora:.2f}%")
assert fora < 0.01, "a classificacao por grupo dominante deixou de ser residual"

mudaram = grupo_linha[grupo_linha["mudou_grupo"]].sort_values("vol_linha", ascending=False)
print(f"\nlinhas com menos de {100 * LIMIAR_MUDOU:.0f}% do volume no grupo dominante: {len(mudaram)} "
      f"({100 * mudaram['vol_linha'].sum() / grupo_linha['vol_linha'].sum():.2f}% do volume)")

# Quando as maiores trocaram de grupo: parcela do volume de cada grupo/lote, por ano
grandes = mudaram.head(6)["linha_blt"]
trajeto = (
    x[x["linha_blt"].isin(grandes)]
    .groupby(["linha_blt", "Ano", "grupo", "lote"], as_index=False)["of_total"].sum()
    .assign(pct_do_ano=lambda d: (100 * d["of_total"] / d.groupby(["linha_blt", "Ano"])["of_total"].transform("sum")).round(1))
    .drop(columns="of_total")
)
print("\nmaiores linhas que trocaram de grupo (parcela do volume do ano em cada grupo/lote):")
print(trajeto.to_string(index=False))

prefixo = grupo_linha["lote"].str.extract(r"^([A-Z]+)")[0].map(PREFIXO_LOTE)
discordam = grupo_linha[prefixo != grupo_linha["grupo"].astype(str)]
print(f"\nlinhas em que o prefixo do lote nao bate com o grupo: {len(discordam)}")
print(discordam[["linha_blt", "grupo", "lote", "empresa"]].to_string(index=False))

volume fora do grupo dominante da propria linha: 0.18%

linhas com menos de 90% do volume no grupo dominante: 26 (0.42% do volume)

maiores linhas que trocaram de grupo (parcela do volume do ano em cada grupo/lote):
linha_blt  Ano                grupo lote  pct_do_ano
  6002-10 2023   Distribuição Local  D10       100.0
  6002-10 2024 Articulação Regional  AR6        80.8
  6002-10 2024   Distribuição Local  D10        19.2
  6026-10 2023   Distribuição Local  D10       100.0
  6026-10 2024   Distribuição Local  D10        18.1
  6026-10 2024           Estrutural  E06        81.9
  6110-10 2023   Distribuição Local  D10       100.0
  6110-10 2024 Articulação Regional  AR6        81.3
  6110-10 2024   Distribuição Local  D10        17.2
  6110-10 2024           Estrutural  E05         1.5
  637V-10 2023   Distribuição Local  D10       100.0
  637V-10 2024   Distribuição Local  D10        18.5
  637V-10 2024           Estrutural  E06        81.5
  677V-10 2023   Distribuição Local  D10  

## Seção 2 — conferência com os arquivos do IQT

Duas conferências contra `Dados/Análise IQT/`:

1. **Lotes**: todo lote que aparece nas linhas está na planilha histórica do IQT (que nomeia os lotes
   sem zero à esquerda: `E1`, não `E01`).
2. **Retrato de 02/09/2026** (`02SETEMBRO2026.xlsx`, mesmo layout da série oficial): quanto da
   classificação de 2023–2024 continua valendo dois anos depois, linha a linha.

In [4]:
# 1. lotes
semzero = lambda lote: re.sub(r"^([A-Z]+)0*(\d)", r"\1\2", lote)
lotes_iqt = {str(l).strip() for l in pd.read_excel(PLANILHA_IQT, sheet_name="IQT Geral", usecols=[0])["LOTE"].dropna()
             if re.fullmatch(r"(E|AR|D)\d+", str(l).strip())}
lotes_nossos = {semzero(l) for l in grupo_linha["lote"].dropna().unique()}
print(f"lotes nas linhas: {len(lotes_nossos)} | lotes na planilha do IQT: {len(lotes_iqt)}")
print("fora da planilha do IQT:", sorted(lotes_nossos - lotes_iqt), "| no IQT e sem linha aqui:", sorted(lotes_iqt - lotes_nossos))

lotes nas linhas: 32 | lotes na planilha do IQT: 32
fora da planilha do IQT: [] | no IQT e sem linha aqui: []


In [5]:
# 2. retrato de 02/09/2026 -- planilha diaria no mesmo layout da serie oficial
bruto = pd.read_excel(SNAPSHOT_IQT, header=None)
cab = bruto.index[bruto.iloc[:, 0].astype(str).str.strip() == "Data"][0]   # a linha do cabecalho nao e fixa
snap = bruto.iloc[cab + 1:].copy()
snap.columns = bruto.iloc[cab].astype(str).str.strip()
col_total = next(c for c in snap.columns if c.startswith("Tot Passageiros"))

snap["linha_blt"] = snap["Linha"].astype(str).str.slice(0, 4) + "-" + snap["Linha"].astype(str).str.slice(4, 6)
snap["grupo_set"] = (snap["Grupo"].astype(str).str.upper().str.replace("Ç", "C").str.replace("Ã", "A").map(GRUPOS))
snap["tot"] = pd.to_numeric(snap[col_total], errors="coerce")
snap = snap.dropna(subset=["grupo_set"]).groupby("linha_blt", as_index=False).agg(
    grupo_set=("grupo_set", "first"), lote_set=("Lote", "first"), tot=("tot", "sum"))

m = snap.merge(grupo_linha[["linha_blt", "grupo", "lote"]], on="linha_blt", how="outer", indicator=True)
novas = m[m["_merge"] == "left_only"]
print(f"linhas em set/2026: {len(snap)} | tambem classificadas aqui: {(m['_merge'] == 'both').sum()} | "
      f"so em set/2026 (criadas depois de 2024): {len(novas)} ({100 * novas['tot'].sum() / snap['tot'].sum():.2f}% do volume do dia)")

ambas = m[m["_merge"] == "both"]
igual = ambas["grupo"].astype(str) == ambas["grupo_set"]
print(f"mesmo grupo: {100 * igual.mean():.1f}% das linhas, {100 * ambas.loc[igual, 'tot'].sum() / ambas['tot'].sum():.1f}% do volume | "
      f"mesmo lote: {100 * (ambas['lote'] == ambas['lote_set']).mean():.1f}% das linhas")
print("\ngrupo 2023-24 (linhas) x grupo em set/2026 (colunas):")
print(pd.crosstab(ambas["grupo"].astype(str), ambas["grupo_set"]).to_string())

linhas em set/2026: 1334 | tambem classificadas aqui: 1312 | so em set/2026 (criadas depois de 2024): 22 (0.87% do volume do dia)
mesmo grupo: 99.5% das linhas, 99.5% do volume | mesmo lote: 99.2% das linhas

grupo 2023-24 (linhas) x grupo em set/2026 (colunas):
grupo_set             Articulação Regional  Distribuição Local  Estrutural
grupo                                                                     
Articulação Regional                   433                   0           0
Distribuição Local                       5                 510           2
Estrutural                               0                   0         362


## Seção 3 — gravação

In [6]:
saida = grupo_linha[["linha_blt", "grupo", "lote", "empresa", "share_grupo_dominante", "mudou_grupo"]]
saida.to_parquet(DIR_SAIDA / "grupo_linha.parquet", index=False)
print(f"grupo_linha.parquet: {len(saida)} linhas")
saida.head()

grupo_linha.parquet: 1469 linhas


,linha_blt,grupo,lote,empresa,share_grupo_dominante,mudou_grupo
0,1012-10,Distribuição Local,D01,Norte Buss,1.0,False
1,1012-21,Distribuição Local,D01,Norte Buss,1.0,False
2,1014-10,Distribuição Local,D01,Norte Buss,1.0,False
3,1015-10,Distribuição Local,D01,Norte Buss,1.0,False
4,1016-10,Distribuição Local,D02,Norte Buss,1.0,False
